# korea-street-imagery-yolo: notebook walkthrough

CLI 스크립트(`scripts/*.py`)와 동일한 파이프라인을 셀 단위로 실행하면서, 수집된 메타데이터와 탐지 결과를 중간중간 표와 이미지로 확인합니다.

별도로 가상환경을 만들거나 `pip install`을 먼저 실행할 필요 없이 바로 시작할 수 있습니다. 바로 아래 셀이 `requirements.txt`의 의존성을 자동으로 설치합니다.

실행 전 확인 사항:
- Python과 Jupyter가 설치되어 있어야 합니다.
- 이 노트북은 `notebooks/run_pipeline.ipynb` 경로 그대로, 즉 프로젝트 루트 아래 `notebooks/` 폴더에 있는 상태로 열어야 합니다.
- `.env` 파일에 `MAPILLARY_ACCESS_TOKEN`이 설정되어 있어야 합니다 (README의 설치 섹션 참고).

In [ ]:
import subprocess
import sys
from pathlib import Path

# 노트북은 notebooks/ 아래에서 실행되므로, 프로젝트 루트를 계산합니다.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# 이미 설치되어 있으면 몇 초 안에 끝납니다. 매번 실행해도 안전합니다.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")],
    check=True,
)

sys.path.insert(0, str(PROJECT_ROOT))
print("의존성 설치 완료. 프로젝트 루트:", PROJECT_ROOT)

In [ ]:
from src.config import load_config
from src.collect import collect_region
from src.detect import detect_images

config = load_config()
print("사용 가능한 지역:", list(config["regions"].keys()))

## 1. 이미지 + 메타데이터 수집 (Mapillary API)

아래에서 `REGION`을 `config.yaml`에 정의된 지역 이름으로 바꿔서 실행하세요.

In [ ]:
REGION = "seoul_gangnam"

meta_df = collect_region(REGION, config)
print(f"수집된 이미지 수: {len(meta_df)}")
meta_df.head()

## 2. YOLO 객체 탐지 실행

방금 수집한 메타데이터를 바로 사용합니다. 이전에 저장해둔 메타데이터 파일을 다시 쓰고 싶다면 `metadata_path`를 `data/metadata/<region>_metadata.parquet`로 바꿔도 됩니다.

In [ ]:
metadata_path = PROJECT_ROOT / config["paths"]["metadata"] / f"{REGION}_metadata.parquet"

det_df = detect_images(metadata_path, config)
print(f"탐지된 객체 수: {len(det_df)}")
det_df.head()

## 3. 클래스별 탐지 개수 확인

In [ ]:
det_df["class_name"].value_counts()

## 4. bounding box 주석 이미지 미리보기

탐지된 이미지 중 하나를 골라 bounding box와 함께 노트북 안에서 바로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

if det_df.empty:
    print("탐지된 객체가 없습니다. REGION이나 config.yaml의 confidence 값을 확인하세요.")
else:
    sample_image_id = det_df.iloc[0]["image_id"]
    annotated_path = PROJECT_ROOT / config["paths"]["annotated_images"] / f"{sample_image_id}.jpg"

    img = Image.open(annotated_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title(sample_image_id)
    plt.show()